# Benchmarking VGGT and VGGT-Omega: Speed, Memory, and Accuracy

This notebook benchmarks two 3D-reconstruction models — **VGGT-1B** and
**VGGT-Omega-1B** — and compares two ways of running them:

- **mixed precision** (the standard way these models ship)
- **bf16** (a lighter-weight mode that halves how much GPU memory the model's
  weights take up)

It answers two practical questions:

- **Speed & memory**: How fast is each model, how much GPU memory does it use as
  you feed it more video frames, and what's the most frames you can process on a
  32GB GPU before running out of memory?
- **Accuracy**: How much precision do you actually lose by switching to bf16 —
  does the camera pose, depth map, or 3D point cloud change noticeably?

Everything below is self-contained — running each cell in order, with the default
settings, reproduces the exact numbers and 3D outputs shared in the accompanying
blog post. Nothing needs to be downloaded or set up separately beyond what's
listed below; the notebook downloads/loads the models itself, runs every
benchmark, and produces the same tables, plots, and glTF 3D files the blog shows.

**Read the "Requirements" and "Things to know before running" sections first** —
a couple of small setup details make a real difference to whether the numbers you
get match the blog.

## Requirements

**Hardware**
- An NVIDIA GPU. This was built and tested on a 32GB RTX 5090; it will run on
  smaller GPUs too, but the frame-count ranges below (especially the "how many
  frames fit" search) assume ~32GB and should be scaled down for smaller cards —
  see `CEILING_SEEDS` in the Configuration cell.
- `nvidia-smi` available on the command line (this ships with any NVIDIA driver).

**Software**
- Python 3.10+, with **`torch` + `torchvision` already installed and matched to
  your own GPU and CUDA version** — get this from
  [pytorch.org/get-started/locally](https://pytorch.org/get-started/locally/).
  VGGT's own listed requirements pin an old torch version — don't blindly
  install from that file, it would downgrade a newer, correctly-matched install
  and could break GPU support outright, especially on newer GPUs.
- `opencv-python`, `numpy`, `pandas`, `huggingface_hub`, `safetensors`,
  `einops`, `trimesh`, `matplotlib`, `scipy`.

**Model source code and weights**
- **VGGT's source code**: public, from
  [github.com/facebookresearch/vggt](https://github.com/facebookresearch/vggt)
  (`git clone` it, then `pip install -r requirements.txt` — skip the
  torch/torchvision lines per above).
- **VGGT-1B's weights**: public and ungated — downloaded automatically from
  Hugging Face (`facebook/VGGT-1B`) the first time the model loads, no setup needed.
- **VGGT-Omega's source code**: public, from
  [github.com/facebookresearch/vggt-omega](https://github.com/facebookresearch/vggt-omega)
  (same clone + install approach).
- **VGGT-Omega's checkpoint**: **gated** on Hugging Face at
  [huggingface.co/facebook/VGGT-Omega](https://huggingface.co/facebook/VGGT-Omega) —
  request access on that page first (an automated approval process, not a
  manual review), then download `vggt_omega_1b_512.pt` from there once
  approved. See the Configuration cell below for exactly where this needs to
  go — it's called out again right where the checkpoint gets loaded. VGGT's
  own results (Part 1 and Part 2) work fine without it; only the Omega half
  needs this file.

**Other files**
- A source video to pull frames from, path set via `VIDEO_PATH` below. Any
  video works — longer videos give more frames to work with at the
  higher-frame-count end of the tests. This isn't provided by the notebook —
  point it at anything you have.

**Time**
- The full run (every frame count, plus the frame-ceiling search for both models
  and both precisions) takes **roughly 1.5–2 hours**. A `QUICK_TEST` flag below
  lets you run everything on a tiny scale first, in a few minutes, just to confirm
  it all works before committing to the full run.

## Things to know before running

A few details that materially affect whether the numbers you get are accurate:

1. **Make sure nothing else is using the GPU while this runs.** GPU memory
   reporting (`nvidia-smi`) counts *everything* running on the card, not just
   this notebook. If another program — a training job, another notebook — is
   using the GPU at the same time, every memory number here will be inflated by
   however much that other program is using. Check first:
   ```bash
   nvidia-smi --query-compute-apps=pid,process_name,used_memory --format=csv
   ```
   If anything shows up there before you start, wait for it to finish or stop it.

2. **GPU memory readings near the limit can wiggle by a GB or so between runs,
   even on an idle GPU — that's normal.** PyTorch keeps a memory cache around
   for speed, and exactly how much it holds onto varies slightly run to run. This
   notebook cross-checks the "how many frames fit" numbers against a second,
   perfectly steady measurement (pure tensor memory, no caching involved) to
   make sure the reported ceiling reflects the model's real behavior and not a
   one-off blip — see the growth-rate check further down.

3. **"bf16" can mean two different things, and they have opposite effects on
   memory.** Just switching on autocast (letting PyTorch cast activations to
   bf16 on the fly, while weights stay in full precision) actually uses *more*
   memory than the standard mode, not less. What this notebook uses instead —
   and what actually saves memory — is casting the model's weights themselves to
   bf16 (`model.to(torch.bfloat16)`), which roughly halves how much space the
   model itself takes up.

4. **VGGT and VGGT-Omega can't be loaded in the same Python session.** They're
   separate codebases with a naming collision in one of their internal modules,
   so this notebook always runs each model in its own subprocess.

5. **Every "how many frames fit" number here is a real measurement, not an
   estimate.** Rather than guessing from a formula, this notebook actually runs
   the model at a candidate frame count, checks whether it fits, and narrows in
   from there — the number you see was actually run, not calculated.

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
# Every path below is relative to WORKSPACE (this notebook's own directory),
# so this works for anyone regardless of where they put things — no
# machine-specific paths. Put the repos/checkpoint/video at these locations
# (siblings of this notebook) and everything resolves automatically.
import os

WORKSPACE = os.path.abspath(".")

# VGGT source: git clone https://github.com/facebookresearch/vggt
REPO_VGGT       = os.path.join(WORKSPACE, "vggt")
# VGGT-Omega source: git clone https://github.com/facebookresearch/vggt-omega
REPO_OMEGA      = os.path.join(WORKSPACE, "vggt-omega")
# VGGT-Omega's checkpoint is GATED on Hugging Face. Request access, then
# download vggt_omega_1b_512.pt, from: https://huggingface.co/facebook/VGGT-Omega
OMEGA_CHECKPOINT = os.path.join(WORKSPACE, "vggt_omega_1b_512.pt")
# VGGT-1B's weights are public/ungated — this auto-downloads on first use from:
# https://huggingface.co/facebook/VGGT-1B
VGGT_HF_REPO    = "facebook/VGGT-1B"
VIDEO_PATH      = os.path.join(WORKSPACE, "11187237-uhd_3840_2160_30fps.mp4")  # <-- point this at your own video

OUT_DIR         = os.path.join(WORKSPACE, "benchmark_out", "notebook_run")
FRAMES_DIR      = os.path.join(OUT_DIR, "frames_pool")
RESULTS_DIR     = os.path.join(OUT_DIR, "results")
WORKERS_DIR     = os.path.join(OUT_DIR, "workers")     # generated worker scripts land here
for _d in (OUT_DIR, FRAMES_DIR, RESULTS_DIR, WORKERS_DIR):
    os.makedirs(_d, exist_ok=True)

# This notebook's DEFAULT settings (QUICK_TEST = False) are the exact settings
# used to produce the numbers published in the blog post — just running every
# cell top to bottom with no changes reproduces those results. Expect roughly
# 1.5-2 hours end to end (Part 1's frame-ceiling search is the slow part).
#
# Set QUICK_TEST = True only if you want to confirm the notebook runs correctly
# on your machine before committing to the full run — it shrinks everything to
# a few minutes, but the numbers it produces are NOT the published results and
# shouldn't be quoted or compared against the blog.
QUICK_TEST = False

N_WARMUP = 1
N_TIMED  = 3
LIMIT_GB = 32.0             # the GPU-capacity target the ceiling search targets
N_QUERY_GRID = 20           # VGGT track_head: N^2 query points

if QUICK_TEST:
    VGGT_SCALE_COUNTS  = [5, 10]
    OMEGA_SCALE_COUNTS = [1, 4, 8]
    VGGT_GAP_COUNTS    = []
    OMEGA_GAP_COUNTS   = []
    CEILING_SEEDS = {("vggt","mixed"): 20, ("vggt","bf16"): 20,
                      ("omega","mixed"): 20, ("omega","bf16"): 20}
    POOL_FRAMES = 40
else:
    # These exact frame counts (including 35, used for the single-frame-count
    # comparison table in the blog) are what produced the published numbers.
    VGGT_SCALE_COUNTS  = [5, 10, 20, 35, 40, 60, 80, 100, 120, 140]
    OMEGA_SCALE_COUNTS = [1, 4, 8, 12, 25, 35, 50, 100, 150, 200]
    VGGT_GAP_COUNTS    = []          # already dense above; add more if needed
    OMEGA_GAP_COUNTS   = [220, 240, 260]
    # Starting probes only (see Part 1 below) — real bisection re-verifies
    # every one of these by actually running it, so on different hardware the
    # search will correctly converge on a different, genuinely measured ceiling
    # rather than reproducing these numbers blindly. These particular values
    # are the exact ceilings found on the reference GPU (a 32GB RTX 5090), used
    # here only to make the search converge quickly on similar hardware.
    CEILING_SEEDS = {("vggt","mixed"): 177, ("vggt","bf16"): 189,
                      ("omega","mixed"): 280, ("omega","bf16"): 331}
    POOL_FRAMES = 450

MAX_FRAMES_HARD_CAP = POOL_FRAMES
MIN_FRAMES_HARD_FLOOR = 1

print(f"QUICK_TEST = {QUICK_TEST}")
print(f"Outputs will land under: {OUT_DIR}")

: 

In [ ]:
# ── Sanity checks — run this before anything else ───────────────────────────
import subprocess, sys

problems = []

if not os.path.isdir(REPO_VGGT):
    problems.append(f"REPO_VGGT does not exist: {REPO_VGGT} "
                     f"(git clone https://github.com/facebookresearch/vggt into this path)")
if not os.path.isdir(REPO_OMEGA):
    problems.append(f"REPO_OMEGA does not exist: {REPO_OMEGA} "
                     f"(git clone https://github.com/facebookresearch/vggt-omega into this path)")
if not os.path.isfile(VIDEO_PATH):
    problems.append(f"Video not found: {VIDEO_PATH} — set VIDEO_PATH in the Configuration cell to point at a real video file")

# Missing Omega checkpoint is a WARNING, not a fatal problem: it's gated on
# Hugging Face (see Requirements above), so it's expected to be missing until
# you've requested and been granted access. VGGT's own results don't need it.
if not os.path.isfile(OMEGA_CHECKPOINT):
    print(f"[WARNING] Omega checkpoint not found at {OMEGA_CHECKPOINT} — it's gated, request "
          f"access at https://huggingface.co/facebook/VGGT-Omega and download vggt_omega_1b_512.pt "
          f"there. VGGT-only cells will still work; anything involving Omega will fail until "
          f"this is resolved.")

try:
    import torch
    if not torch.cuda.is_available():
        problems.append("torch.cuda.is_available() is False — no usable GPU visible to torch")
    else:
        print(f"torch {torch.__version__}  |  CUDA {torch.version.cuda}  |  "
              f"GPU: {torch.cuda.get_device_name(0)}  |  "
              f"{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB total")
except ImportError:
    problems.append("torch is not importable in this kernel")

try:
    gpu_procs = subprocess.check_output(
        ["nvidia-smi", "--query-compute-apps=pid,process_name,used_memory",
         "--format=csv,noheader"], stderr=subprocess.DEVNULL).decode().strip()
    if gpu_procs:
        print("\n[WARNING] GPU is NOT exclusive right now — other process(es) using it:")
        print(gpu_procs)
        print("Every memory reading below will be inflated by however much these use.")
    else:
        print("\nGPU is idle (no other compute processes) — good to proceed.")
except FileNotFoundError:
    problems.append("nvidia-smi not found on PATH")

if problems:
    print("\n[PROBLEMS FOUND]")
    for p in problems:
        print(" -", p)
    raise SystemExit("Fix the above before continuing.")
else:
    print("\nAll checks passed.")

---
# Part 1 — Speed, Memory, and How Many Frames Fit

## Why every measurement runs in its own fresh process

Each individual measurement below (one model, one precision setting, one frame
count) starts a brand-new Python process rather than reusing the same one for
everything. This matters for accuracy: PyTorch keeps some GPU memory reserved
between calls for efficiency, and that reservation can grow the more you reuse a
process for different-sized workloads — which would quietly inflate the memory
numbers for later measurements. Starting fresh each time means every reading
reflects only what that one measurement actually needed.

The next cell writes a small worker script to disk that does exactly one
measurement per run — that's the file this notebook will launch, over and over,
with different settings each time.

In [ ]:
# ── Write the isolated worker script to disk ────────────────────────────────
# This is the exact logic validated in precision_bench/unified_worker.py,
# parameterized here via environment variables so it stays a standalone file
# with no import dependency on this notebook or on precision_bench/.

WORKER_SRC = r"""
import os, sys, argparse, gc, json, time, threading, subprocess

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

REPO_VGGT  = os.environ["NB_REPO_VGGT"]
REPO_OMEGA = os.environ["NB_REPO_OMEGA"]
OMEGA_CKPT = os.environ["NB_OMEGA_CKPT"]
VGGT_HF    = os.environ["NB_VGGT_HF"]
N_QUERY_GRID = int(os.environ.get("NB_N_QUERY_GRID", "20"))


def _global_used_mib():
    out = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits", "-i", "0"],
        stderr=subprocess.DEVNULL).decode().strip().splitlines()[0]
    return float(out)


def _self_used_mib(pid):
    # GPU memory attributed to THIS process only (immune to other processes).
    try:
        out = subprocess.check_output(
            ["nvidia-smi", "--query-compute-apps=pid,used_memory",
             "--format=csv,noheader,nounits", "-i", "0"], stderr=subprocess.DEVNULL).decode()
        for line in out.strip().splitlines():
            parts = [p.strip() for p in line.split(",")]
            if len(parts) >= 2 and parts[0].isdigit() and int(parts[0]) == pid:
                return float(parts[1])
    except Exception:
        pass
    return 0.0


class Sampler(threading.Thread):
    # Background nvidia-smi peak memory sampler.
    # Tracks the GPU-global peak (what actually OOMs the card) AND a per-pid peak
    # (contamination-proof, for diagnosing whether another process polluted the run).
    def __init__(self, interval=0.05):
        super().__init__(daemon=True)
        self.interval = interval
        self.peak_mib = 0.0
        self.self_peak_mib = 0.0
        self._pid = os.getpid()
        self._stop = False

    def run(self):
        while not self._stop:
            try:
                self.peak_mib = max(self.peak_mib, _global_used_mib())
            except Exception:
                pass
            self.self_peak_mib = max(self.self_peak_mib, _self_used_mib(self._pid))
            time.sleep(self.interval)

    def stop(self):
        self._stop = True
        self.join(timeout=1)


def load_frame_paths(frames_dir, n):
    paths = sorted(
        os.path.join(frames_dir, f) for f in os.listdir(frames_dir) if f.endswith(".png")
    )
    if len(paths) < n:
        raise RuntimeError(f"only {len(paths)} frames available in {frames_dir}, need {n}")
    return paths[:n]


def _timer(t_acc, m_acc, reset):
    import torch
    def timed(name, fn):
        torch.cuda.synchronize()
        if reset:
            torch.cuda.reset_peak_memory_stats()
        t0 = time.perf_counter()
        out = fn()
        torch.cuda.synchronize()
        t_acc.setdefault(name, []).append(time.perf_counter() - t0)
        m_acc[name] = max(m_acc.get(name, 0.0), torch.cuda.max_memory_allocated() / 1e9)
        return out
    return timed


def build_vggt(precision, paths):
    import torch
    sys.path.insert(0, REPO_VGGT)
    from vggt.models.vggt import VGGT
    from vggt.utils.load_fn import load_and_preprocess_images

    t0 = time.perf_counter()
    # Public, ungated weights -- auto-downloads from https://huggingface.co/facebook/VGGT-1B
    model = VGGT.from_pretrained(VGGT_HF).eval().to("cuda")
    torch.cuda.synchronize()
    load_s = time.perf_counter() - t0

    images = load_and_preprocess_images(paths).to("cuda")
    images_b = images.unsqueeze(0)
    S, _, H, W = images.shape
    xs = torch.linspace(0.05 * W, 0.95 * W, N_QUERY_GRID, device="cuda")
    ys = torch.linspace(0.05 * H, 0.95 * H, N_QUERY_GRID, device="cuda")
    gx, gy = torch.meshgrid(xs, ys, indexing="xy")
    qp = torch.stack([gx.reshape(-1), gy.reshape(-1)], -1).unsqueeze(0)
    dtype = torch.bfloat16

    # TRUE bf16: cast model weights AND inputs to bf16 (halves resident weight
    # memory). Autocast stays on so fp32-hardcoded positional embeddings in the
    # DPT/dense heads don't crash a pure-bf16 pass. Query-point coords stay fp32.
    if precision == "bf16":
        model = model.to(torch.bfloat16)
        images_b = images_b.to(torch.bfloat16)

    def fwd(t_acc, m_acc, reset=True):
        timed = _timer(t_acc, m_acc, reset)
        with torch.no_grad():
            if precision == "mixed":
                with torch.autocast("cuda", dtype=dtype):
                    tok, ps = timed("aggregator", lambda: model.aggregator(images_b))
                with torch.autocast("cuda", enabled=False):
                    timed("camera_head", lambda: model.camera_head(tok)[-1])
                    timed("depth_head", lambda: model.depth_head(tok, images=images_b, patch_start_idx=ps))
                    timed("point_head", lambda: model.point_head(tok, images=images_b, patch_start_idx=ps))
                with torch.autocast("cuda", dtype=dtype):
                    timed("track_head", lambda: model.track_head(
                        tok, images=images_b, patch_start_idx=ps, query_points=qp))
            else:
                with torch.autocast("cuda", dtype=dtype):
                    tok, ps = timed("aggregator", lambda: model.aggregator(images_b))
                    timed("camera_head", lambda: model.camera_head(tok)[-1])
                    timed("depth_head", lambda: model.depth_head(tok, images=images_b, patch_start_idx=ps))
                    timed("point_head", lambda: model.point_head(tok, images=images_b, patch_start_idx=ps))
                    timed("track_head", lambda: model.track_head(
                        tok, images=images_b, patch_start_idx=ps, query_points=qp))

    return fwd, load_s, S, H, W


def build_omega(precision, paths):
    import torch
    sys.path.insert(0, REPO_OMEGA)
    from vggt_omega.models import VGGTOmega
    from vggt_omega.utils.load_fn import load_and_preprocess_images

    t0 = time.perf_counter()
    model = VGGTOmega().to("cuda").eval()
    # GATED checkpoint -- request access at https://huggingface.co/facebook/VGGT-Omega,
    # then download vggt_omega_1b_512.pt there once approved.
    model.load_state_dict(
        torch.load(OMEGA_CKPT, map_location="cpu", weights_only=False), strict=False)
    torch.cuda.synchronize()
    load_s = time.perf_counter() - t0

    images = load_and_preprocess_images(paths, image_resolution=512).to("cuda")
    images_b = images.unsqueeze(0)
    S, _, H, W = images.shape
    dtype = torch.bfloat16

    if precision == "bf16":
        model = model.to(torch.bfloat16)
        images_b = images_b.to(torch.bfloat16)

    def fwd(t_acc, m_acc, reset=True):
        timed = _timer(t_acc, m_acc, reset)
        with torch.no_grad():
            if precision == "mixed":
                with torch.autocast("cuda", dtype=dtype):
                    tok, ps = timed("aggregator", lambda: model.aggregator(images_b))
                with torch.autocast("cuda", enabled=False):
                    timed("camera_head", lambda: model.camera_head(tok, patch_token_start=ps))
                    timed("dense_head", lambda: model.dense_head(tok, images=images_b, patch_token_start=ps))
            else:
                with torch.autocast("cuda", dtype=dtype):
                    tok, ps = timed("aggregator", lambda: model.aggregator(images_b))
                    timed("camera_head", lambda: model.camera_head(tok, patch_token_start=ps))
                    timed("dense_head", lambda: model.dense_head(tok, images=images_b, patch_token_start=ps))

    return fwd, load_s, S, H, W


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", choices=["vggt", "omega"], required=True)
    ap.add_argument("--precision", choices=["mixed", "bf16"], required=True)
    ap.add_argument("--frames", type=int, required=True)
    ap.add_argument("--frames-dir", required=True)
    ap.add_argument("--warmup", type=int, default=1)
    ap.add_argument("--timed", type=int, default=3)
    args = ap.parse_args()

    import torch

    result = dict(model=args.model, precision=args.precision, frames=args.frames)
    try:
        baseline_mib = 0.0
        try:
            baseline_mib = _global_used_mib()
        except Exception:
            pass
        result["nvsmi_baseline_gb"] = round(baseline_mib * (2 ** 20) / 1e9, 4)

        paths = load_frame_paths(args.frames_dir, args.frames)
        gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()

        builder = build_vggt if args.model == "vggt" else build_omega
        fwd, load_s, S, H, W = builder(args.precision, paths)
        result["model_load_s"] = round(load_s, 2)
        result["resolution_hw"] = [int(H), int(W)]
        result["n_frames_actual"] = int(S)

        for _ in range(args.warmup):
            fwd({}, {})
        torch.cuda.synchronize()

        t_acc, m_acc = {}, {}
        for _ in range(args.timed):
            gc.collect(); torch.cuda.empty_cache(); torch.cuda.synchronize()
            fwd(t_acc, m_acc)

        # Clean whole-forward peak pass: nvidia-smi sampled concurrently, this is
        # the ONLY number that counts as "the" memory measurement for this run.
        gc.collect(); torch.cuda.empty_cache(); torch.cuda.synchronize()
        sampler = Sampler(); sampler.start()
        torch.cuda.reset_peak_memory_stats()
        t0 = time.perf_counter()
        fwd({}, {}, reset=False)
        torch.cuda.synchronize()
        clean_s = time.perf_counter() - t0
        sampler.stop()

        timings = {k: sum(v) / len(v) for k, v in t_acc.items()}
        fwd_total = sum(timings.values())

        result.update(
            oom=False,
            timings_s={k: round(v, 5) for k, v in timings.items()},
            forward_total_s=round(fwd_total, 4),
            ms_per_frame=round(fwd_total / S * 1000, 2),
            fps=round(S / fwd_total, 2),
            clean_pass_s=round(clean_s, 4),
            torch_peak_alloc_gb=round(torch.cuda.max_memory_allocated() / 1e9, 4),
            torch_peak_reserved_gb=round(torch.cuda.max_memory_reserved() / 1e9, 4),
            nvsmi_peak_gb=round(sampler.peak_mib * (2 ** 20) / 1e9, 4),
            nvsmi_self_peak_gb=round(sampler.self_peak_mib * (2 ** 20) / 1e9, 4),
            contaminated=bool(baseline_mib * (2 ** 20) / 1e9 > 1.5),
            torch_version=torch.__version__,
            cuda_version=torch.version.cuda,
            gpu_name=torch.cuda.get_device_name(0),
            gpu_total_gb=round(torch.cuda.get_device_properties(0).total_memory / 1e9, 3),
        )
    except torch.cuda.OutOfMemoryError as e:
        result.update(oom=True, error=str(e)[:300])
    except Exception as e:
        import traceback
        result.update(oom=False, error=f"EXCEPTION: {e}", traceback=traceback.format_exc()[:2000])

    print("RESULT_JSON:" + json.dumps(result), flush=True)


if __name__ == "__main__":
    main()
"""

WORKER_PATH = os.path.join(WORKERS_DIR, "worker_unified.py")
with open(WORKER_PATH, "w") as f:
    f.write(WORKER_SRC)
print(f"Wrote worker script -> {WORKER_PATH}")

### What that worker script does, each time it's run

1. Loads the requested model, and if bf16 was requested, casts both the model
   and its inputs to bf16.
2. Runs a warmup pass (so one-time setup costs don't skew the timing), then a
   few timed passes — the average of those is the reported inference time,
   broken down by each stage of the model (backbone, camera, depth, etc).
3. Runs one more pass while measuring GPU memory, and reports the peak.
4. Reports the result. If the GPU runs out of memory, that's recorded as a
   real result too (not treated as a crash) — it's exactly the information the
   frame-ceiling search needs.

In [ ]:
# ── Orchestration: call the worker, cache results, resumable state ─────────
import json, time, subprocess, sys as _sys

RESULTS_JSON = os.path.join(RESULTS_DIR, "notebook_results.json")
LOG_PATH     = os.path.join(RESULTS_DIR, "notebook_run.log")
WORKER_TIMEOUT_S = 1200

def log(msg):
    line = f"[{time.strftime('%H:%M:%S')}] {msg}"
    print(line, flush=True)
    with open(LOG_PATH, "a") as f:
        f.write(line + "\n")

def load_state():
    if os.path.exists(RESULTS_JSON):
        return json.load(open(RESULTS_JSON))
    return {"runs": {}, "ceilings": {}}

def save_state(state):
    tmp = RESULTS_JSON + ".tmp"
    json.dump(state, open(tmp, "w"), indent=2)
    os.replace(tmp, RESULTS_JSON)

def run_key(model, precision, frames):
    return f"{model}_{precision}_{frames}f"

def call_worker(model, precision, frames):
    """Launch the worker as a FRESH subprocess — never call its functions directly."""
    env = os.environ.copy()
    env["NB_REPO_VGGT"] = REPO_VGGT
    env["NB_REPO_OMEGA"] = REPO_OMEGA
    env["NB_OMEGA_CKPT"] = OMEGA_CHECKPOINT
    env["NB_VGGT_HF"] = VGGT_HF_REPO
    env["NB_N_QUERY_GRID"] = str(N_QUERY_GRID)
    cmd = [_sys.executable, WORKER_PATH, "--model", model, "--precision", precision,
           "--frames", str(frames), "--frames-dir", FRAMES_DIR,
           "--warmup", str(N_WARMUP), "--timed", str(N_TIMED)]
    t0 = time.time()
    try:
        proc = subprocess.run(cmd, capture_output=True, text=True, timeout=WORKER_TIMEOUT_S, env=env)
    except subprocess.TimeoutExpired:
        elapsed = time.time() - t0
        log(f"  [{model}/{precision}] {frames:4d}f  TIMEOUT after {WORKER_TIMEOUT_S}s")
        return dict(model=model, precision=precision, frames=frames, oom=False,
                    error="TIMEOUT", elapsed_wall_s=round(elapsed, 1))
    elapsed = time.time() - t0
    line = None
    for l in proc.stdout.splitlines():
        if l.startswith("RESULT_JSON:"):
            line = l[len("RESULT_JSON:"):]
    if line is None:
        tail = "\n".join(proc.stderr.splitlines()[-20:])
        log(f"  [{model}/{precision}] {frames:4d}f  CRASH (exit {proc.returncode}): {tail[-200:]}")
        return dict(model=model, precision=precision, frames=frames, oom=True,
                    error=f"CRASH exit={proc.returncode}", stderr_tail=tail,
                    elapsed_wall_s=round(elapsed, 1))
    r = json.loads(line)
    r["elapsed_wall_s"] = round(elapsed, 1)
    if r.get("oom"):
        log(f"  [{model}/{precision}] {frames:4d}f  OOM  (wall {elapsed:.0f}s)")
    elif r.get("error"):
        log(f"  [{model}/{precision}] {frames:4d}f  ERROR: {r['error'][:120]}  (wall {elapsed:.0f}s)")
    else:
        log(f"  [{model}/{precision}] {frames:4d}f  {r['forward_total_s']:.3f}s "
            f"({r['fps']:.1f}fps, {r['ms_per_frame']:.1f}ms/f)  "
            f"nvsmi={r['nvsmi_peak_gb']:.3f}GB  (wall {elapsed:.0f}s)")
    return r

def measure(state, model, precision, frames):
    """Cached-or-fresh measurement. Only trusts the cache for successful runs,
    so a crash/OOM/timeout is automatically retried next time this is called
    with the same arguments — this is what makes the whole run resumable."""
    k = run_key(model, precision, frames)
    cached = state["runs"].get(k)
    if cached is not None and not cached.get("error") and not cached.get("oom"):
        return cached
    r = call_worker(model, precision, frames)
    state["runs"][k] = r
    save_state(state)
    return r

state = load_state()
print("Orchestration helpers ready. State file:", RESULTS_JSON)
print(f"{len(state['runs'])} measurements already cached from a previous run (if any).")

### Why results are saved after every measurement

The full run takes 1.5–2 hours, so it's worth being able to pick back up where
you left off if it's interrupted. `measure()` above checks whether a given
`(model, precision, frame count)` has already been measured before running it
again — so if you re-run this notebook's cells, anything already done is
skipped, and only what's missing gets computed.

In [ ]:
# ── Extract the frame pool ───────────────────────────────────────────────────
# One shared pool of frames, uniformly sampled across the whole source video,
# large enough to serve every frame count this notebook will ever request
# (scaling table AND ceiling search). Extracted ONCE; every measurement above
# takes a PREFIX of this pool (paths[:n]) so frame N is identical across every
# run that uses at least N frames — this keeps the comparison fair.
import cv2

def ensure_frame_pool():
    existing = sorted(f for f in os.listdir(FRAMES_DIR) if f.endswith(".png"))
    if len(existing) >= POOL_FRAMES:
        print(f"{len(existing)} frames already in the pool — skipping extraction.")
        return
    for f in existing:
        os.remove(os.path.join(FRAMES_DIR, f))
    cap = cv2.VideoCapture(VIDEO_PATH)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total < POOL_FRAMES:
        raise RuntimeError(f"source video only has {total} frames, need {POOL_FRAMES}")
    idxs = [int(round(i * (total - 1) / (POOL_FRAMES - 1))) for i in range(POOL_FRAMES)]
    saved = 0
    for j, idx in enumerate(idxs):
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ok, frame = cap.read()
        if ok:
            cv2.imwrite(os.path.join(FRAMES_DIR, f"{j:04d}.png"), frame)
            saved += 1
    cap.release()
    print(f"Extracted {saved}/{POOL_FRAMES} frames from {os.path.basename(VIDEO_PATH)} "
          f"({total} source frames, uniform sampling).")

ensure_frame_pool()

## 1.1 — Speed and memory at different frame counts

This runs both models, in both precisions, at a range of frame counts, to see
how inference time and GPU memory scale as you feed the model more frames.
**This is the first cell that actually uses the GPU for a while** — a couple of
minutes under `QUICK_TEST`, tens of minutes for the full run (the frame-ceiling
search after this is the slower part).

In [ ]:
for model, counts in (("vggt", VGGT_SCALE_COUNTS), ("omega", OMEGA_SCALE_COUNTS)):
    for precision in ("mixed", "bf16"):
        log(f"--- Scaling table: {model}/{precision} ---")
        for n in counts:
            measure(state, model, precision, n)

for model, counts in (("vggt", VGGT_GAP_COUNTS), ("omega", OMEGA_GAP_COUNTS)):
    for precision in ("mixed", "bf16"):
        if not counts:
            continue
        log(f"--- Gap-fill: {model}/{precision} ---")
        for n in counts:
            measure(state, model, precision, n)

print("Scaling tables complete.")

## 1.2 — Finding the maximum number of frames that fit in 32GB

For each model and precision, this searches for the largest number of frames
that can be processed without running out of GPU memory. It works by actually
running the model, not by estimating from a formula:

1. Start from a rough guess.
2. If that guess still fits, try progressively larger frame counts until one
   doesn't fit (or the GPU runs out of memory).
3. Once you have one frame count that fits and one that doesn't, narrow in
   between them — each attempt is a real run — until you've found the exact
   boundary: the largest frame count that fits, and the smallest one that doesn't.

Every number produced by this search is something that was actually run and
measured, not calculated.

In [ ]:
def is_over(r):
    if r.get("oom") or r.get("error"):
        return True
    return r.get("nvsmi_peak_gb", 0.0) > LIMIT_GB


def find_ceiling(state, model, precision, seed):
    log(f"=== Ceiling search: {model}/{precision}  seed(starting probe)={seed}f ===")
    tested = {}

    def test(n):
        n = max(MIN_FRAMES_HARD_FLOOR, min(MAX_FRAMES_HARD_CAP, int(n)))
        if n in tested:
            return n, tested[n]
        r = measure(state, model, precision, n)
        tested[n] = r
        return n, r

    n0, r0 = test(seed)
    lo = hi = None

    if not is_over(r0):
        lo = n0
        step = max(2, seed // 8)
        n = n0
        while hi is None:
            n2 = n + step
            if n2 >= MAX_FRAMES_HARD_CAP:
                ncap, rcap = test(MAX_FRAMES_HARD_CAP)
                if not is_over(rcap):
                    log(f"  reached hard cap {MAX_FRAMES_HARD_CAP}f without crossing "
                        f"{LIMIT_GB} GB — no ceiling within probed range")
                    return dict(ceiling_frames=ncap, ceiling_nvsmi_gb=rcap.get("nvsmi_peak_gb"),
                                first_over_frames=None, first_over_nvsmi_gb=None,
                                note=f"hard cap {MAX_FRAMES_HARD_CAP}f reached, still under limit")
                lo, hi = n, ncap
                break
            n2i, r2 = test(n2)
            if is_over(r2):
                hi = n2i
            else:
                lo = n2i
                n = n2i
                step *= 2
    else:
        hi = n0
        step = max(2, seed // 8)
        n = n0
        while lo is None:
            n2 = n - step
            if n2 <= MIN_FRAMES_HARD_FLOOR:
                nfloor, rfloor = test(MIN_FRAMES_HARD_FLOOR)
                if is_over(rfloor):
                    log(f"  even {MIN_FRAMES_HARD_FLOOR}f exceeds {LIMIT_GB} GB — no valid ceiling")
                    return dict(ceiling_frames=None, ceiling_nvsmi_gb=None,
                                first_over_frames=nfloor,
                                first_over_nvsmi_gb=rfloor.get("nvsmi_peak_gb"),
                                note="no frame count under the limit was found")
                lo = nfloor
                break
            n2i, r2 = test(n2)
            if is_over(r2):
                hi = n2i
                n = n2i
                step *= 2
            else:
                lo = n2i

    while hi - lo > 1:
        mid = (lo + hi) // 2
        _, rmid = test(mid)
        if is_over(rmid):
            hi = mid
        else:
            lo = mid

    r_lo, r_hi = tested[lo], tested[hi]
    log(f"  -> MEASURED ceiling = {lo}f (nvsmi={r_lo.get('nvsmi_peak_gb')} GB)  |  "
        f"MEASURED first-over = {hi}f "
        f"({'OOM' if r_hi.get('oom') else str(r_hi.get('nvsmi_peak_gb')) + ' GB'})")
    return dict(ceiling_frames=lo, ceiling_nvsmi_gb=r_lo.get("nvsmi_peak_gb"),
                first_over_frames=hi, first_over_nvsmi_gb=r_hi.get("nvsmi_peak_gb"),
                first_over_oom=bool(r_hi.get("oom")), probes=sorted(tested.keys()))


for (model, precision), seed in CEILING_SEEDS.items():
    key = f"{model}_{precision}"
    if key in state["ceilings"] and state["ceilings"][key].get("ceiling_frames") is not None:
        log(f"[ceiling] {key} already resolved -> {state['ceilings'][key]}")
        continue
    result = find_ceiling(state, model, precision, seed)
    state["ceilings"][key] = result
    save_state(state)

print("Ceiling searches complete.")

## 1.3 — Results: tables and plots

Everything below just reads back what was already measured above — no more GPU
work happens from here on in this part.

In [ ]:
import pandas as pd

def measured_points(model, precision):
    pts = []
    for k, r in state["runs"].items():
        if k.startswith(f"{model}_{precision}_") and not r.get("oom") and not r.get("error"):
            n = int(k.rsplit("_", 1)[-1][:-1])
            pts.append((n, r))
    return sorted(pts)

rows = []
for model in ("vggt", "omega"):
    for precision in ("mixed", "bf16"):
        for n, r in measured_points(model, precision):
            rows.append(dict(
                model=model, precision=precision, frames=n,
                resolution=f"{r['resolution_hw'][0]}x{r['resolution_hw'][1]}",
                inference_s=r["forward_total_s"], ms_per_frame=r["ms_per_frame"],
                fps=r["fps"], nvsmi_peak_gb=r["nvsmi_peak_gb"],
                torch_allocated_gb=r["torch_peak_alloc_gb"],
            ))
scaling_df = pd.DataFrame(rows).sort_values(["model", "precision", "frames"])
scaling_df

In [ ]:
ceiling_rows = []
labels = {"vggt": "VGGT-1B", "omega": "VGGT-Omega-1B-512"}
for (model, precision) in CEILING_SEEDS:
    key = f"{model}_{precision}"
    c = state["ceilings"].get(key, {})
    ceiling_rows.append(dict(
        model=labels[model], precision=precision,
        ceiling_frames=c.get("ceiling_frames"), ceiling_nvsmi_gb=c.get("ceiling_nvsmi_gb"),
        first_over_frames=c.get("first_over_frames"),
        first_over_nvsmi_gb=c.get("first_over_nvsmi_gb"),
    ))
ceiling_df = pd.DataFrame(ceiling_rows)
ceiling_df

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for ax, model in zip(axes[0], ("vggt", "omega")):
    for precision, color in (("mixed", "tab:blue"), ("bf16", "tab:orange")):
        d = scaling_df[(scaling_df.model == model) & (scaling_df.precision == precision)]
        ax.plot(d.frames, d.nvsmi_peak_gb, "o-", color=color, label=f"{precision} (nvidia-smi)")
    ax.axhline(LIMIT_GB, color="red", ls="--", lw=1, label=f"{LIMIT_GB} GB limit")
    ax.set(title=f"{labels[model]} — GPU memory vs frames", xlabel="frames", ylabel="GB")
    ax.legend(); ax.grid(alpha=.3)

for ax, model in zip(axes[1], ("vggt", "omega")):
    for precision, color in (("mixed", "tab:blue"), ("bf16", "tab:orange")):
        d = scaling_df[(scaling_df.model == model) & (scaling_df.precision == precision)]
        ax.plot(d.frames, d.ms_per_frame, "o-", color=color, label=precision)
    ax.set(title=f"{labels[model]} — ms/frame vs frames", xlabel="frames", ylabel="ms/frame")
    ax.legend(); ax.grid(alpha=.3)

fig.tight_layout()
plt.show()

## 1.4 — Double-checking the memory numbers

The memory readings in the plots above (from `nvidia-smi`) can look a little
jumpy near the 32GB limit — that's expected (see "Things to know before
running," point 2), not a sign that something's wrong. To confirm the model's
actual memory usage really is growing smoothly, the cell below looks at a second,
noise-free measurement — the raw size of the tensors PyTorch is holding — and
shows it increases by essentially the same fixed amount per frame the entire
way through. That's the real underlying trend the `nvidia-smi` readings are
wobbling around.

In [ ]:
def steady_state_alloc_slope(model, precision, tolerance=0.07):
    """Median GB/frame growth rate of torch_peak_allocated_gb, with low-frame
    kernel/algorithm-selection transients excluded via outlier filtering (not a
    fixed skip-count, since different models settle at different points)."""
    pts = measured_points(model, precision)
    slopes = []
    for (n0, r0), (n1, r1) in zip(pts, pts[1:]):
        d = n1 - n0
        if d <= 0:
            continue
        slopes.append((r1["torch_peak_alloc_gb"] - r0["torch_peak_alloc_gb"]) / d)
    if not slopes:
        return None
    rough_median = sorted(slopes)[len(slopes) // 2]
    steady = [s for s in slopes if abs(s - rough_median) <= tolerance * rough_median] or slopes
    steady_sorted = sorted(steady)
    return dict(median=steady_sorted[len(steady_sorted) // 2], min=min(steady), max=max(steady),
                n=len(steady), excluded=len(slopes) - len(steady))

growth_rows = []
for model in ("vggt", "omega"):
    for precision in ("mixed", "bf16"):
        sl = steady_state_alloc_slope(model, precision)
        if sl:
            growth_rows.append(dict(model=labels[model], precision=precision,
                                     growth_rate_GB_per_frame=round(sl["median"], 5),
                                     range_min=round(sl["min"], 5), range_max=round(sl["max"], 5),
                                     intervals_used=sl["n"], transients_excluded=sl["excluded"]))
growth_df = pd.DataFrame(growth_rows)
growth_df

The `range_min`/`range_max` columns show just how consistent this is — if
memory use weren't growing smoothly, these would vary a lot across the frame
range. Instead they barely move at all, from the smallest frame count tested
all the way up to the measured limit.

## 1.5 — Troubleshooting

- **A run reports running out of memory much earlier than expected**: check the
  `contaminated` field in that result — if `True`, something else was using the
  GPU at the time. Confirm the GPU is idle and re-run.
- **The frame-ceiling search takes a while**: at high frame counts, close to a
  GPU's limit, a single run can take tens of seconds, and each measurement
  involves several passes. This is expected, not a bug.
- **"only N frames available, need M"**: raise `POOL_FRAMES` in the
  Configuration cell and re-run `ensure_frame_pool()`.
- **Numbers differ from a previous run**: check `torch_version` and `gpu_name`
  in the results — different hardware or software versions will legitimately
  produce different numbers. These results are specific to the machine they
  were run on, not universal constants.

---
# Part 2 — How Much Accuracy Does bf16 Actually Cost?

Part 1 covered speed and memory. This part answers a different question: **how
much does the 3D reconstruction itself change** when you switch from the
standard mixed precision to bf16 — does the camera position drift, does the
depth map change, does the point cloud shift?

Both models process the same set of real video frames here, at both
precisions — four reconstructions in total: VGGT mixed, VGGT bf16, Omega mixed,
Omega bf16 — each exported as its own real `.gltf`/`.glb` 3D file, so you can
open all four yourself and compare them side by side.

As in Part 1, each model runs in its own subprocess — here it's because VGGT
and VGGT-Omega are separate codebases that can't both be loaded at once.

In [ ]:
# ── Write the shared scene/glTF export helper ───────────────────────────────
# Identical point-cloud + camera-frustum builder used for BOTH models, so the
# two reconstructions are filtered and exported the same way. Copied verbatim
# from precision_bench/scene.py (camera-mesh helper functions originate in
# vggt/visual_util.py).
SCENE_SRC = r"""
import numpy as np
import trimesh
import matplotlib
from scipy.spatial.transform import Rotation


def integrate_camera_into_scene(scene, transform, face_colors, scene_scale):
    cam_width = scene_scale * 0.05
    cam_height = scene_scale * 0.1
    rot_45_degree = np.eye(4)
    rot_45_degree[:3, :3] = Rotation.from_euler("z", 45, degrees=True).as_matrix()
    rot_45_degree[2, 3] = -cam_height
    opengl_transform = get_opengl_conversion_matrix()
    complete_transform = transform @ opengl_transform @ rot_45_degree
    camera_cone_shape = trimesh.creation.cone(cam_width, cam_height, sections=4)
    slight_rotation = np.eye(4)
    slight_rotation[:3, :3] = Rotation.from_euler("z", 2, degrees=True).as_matrix()
    vertices_combined = np.concatenate([
        camera_cone_shape.vertices,
        0.95 * camera_cone_shape.vertices,
        transform_points(slight_rotation, camera_cone_shape.vertices),
    ])
    vertices_transformed = transform_points(complete_transform, vertices_combined)
    mesh_faces = compute_camera_faces(camera_cone_shape)
    camera_mesh = trimesh.Trimesh(vertices=vertices_transformed, faces=mesh_faces)
    camera_mesh.visual.face_colors[:, :3] = face_colors
    scene.add_geometry(camera_mesh)


def apply_scene_alignment(scene_3d, extrinsics_matrices):
    opengl_conversion_matrix = get_opengl_conversion_matrix()
    align_rotation = np.eye(4)
    align_rotation[:3, :3] = Rotation.from_euler("y", 180, degrees=True).as_matrix()
    initial_transformation = np.linalg.inv(extrinsics_matrices[0]) @ opengl_conversion_matrix @ align_rotation
    scene_3d.apply_transform(initial_transformation)
    return scene_3d


def get_opengl_conversion_matrix():
    matrix = np.identity(4)
    matrix[1, 1] = -1
    matrix[2, 2] = -1
    return matrix


def transform_points(transformation, points, dim=None):
    points = np.asarray(points)
    initial_shape = points.shape[:-1]
    dim = dim or points.shape[-1]
    transformation = transformation.swapaxes(-1, -2)
    points = points @ transformation[..., :-1, :] + transformation[..., -1:, :]
    return points[..., :dim].reshape(*initial_shape, dim)


def build_scene(points, conf, images_np, extrinsic, conf_thres_pct, max_points, seed=0):
    verts = points.reshape(-1, 3)
    colors = (np.transpose(images_np, (0, 2, 3, 1)).reshape(-1, 3) * 255).astype(np.uint8)
    c = conf.reshape(-1)
    thr = 0.0 if conf_thres_pct == 0.0 else np.percentile(c, conf_thres_pct)
    mask = (c >= thr) & (c > 1e-5) & np.isfinite(verts).all(axis=1)
    idx = np.where(mask)[0]
    n_conf = idx.size
    if max_points and idx.size > max_points:
        idx = np.random.default_rng(seed).choice(idx, size=max_points, replace=False)
    verts, colors = verts[idx], colors[idx]
    if verts.size == 0:
        verts = np.array([[0.0, 0.0, 0.0]]); colors = np.array([[255, 255, 255]], np.uint8)
        scene_scale = 1.0
    else:
        lo, hi = np.percentile(verts, 5, axis=0), np.percentile(verts, 95, axis=0)
        scene_scale = float(np.linalg.norm(hi - lo)) or 1.0
    scene = trimesh.Scene()
    scene.add_geometry(trimesh.PointCloud(vertices=verts, colors=colors))
    S = extrinsic.shape[0]
    ext4 = np.zeros((S, 4, 4)); ext4[:, :3, :4] = extrinsic; ext4[:, 3, 3] = 1.0
    cmap = matplotlib.colormaps.get_cmap("gist_rainbow")
    for i in range(S):
        c2w = np.linalg.inv(ext4[i])
        col = tuple(int(255 * x) for x in cmap(i / max(S, 1))[:3])
        integrate_camera_into_scene(scene, c2w, col, scene_scale)
    scene = apply_scene_alignment(scene, ext4)
    return scene, int(len(verts)), int(n_conf)


def export_gltf_file(scene, out_path):
    import os
    os.makedirs(os.path.dirname(out_path) or ".", exist_ok=True)
    files = trimesh.exchange.gltf.export_gltf(scene, embed_buffers=True)
    data = next(v for k, v in files.items() if k.endswith(".gltf"))
    with open(out_path, "wb") as f:
        f.write(data)
    glb_path = os.path.splitext(out_path)[0] + ".glb"
    scene.export(glb_path)
    return out_path, glb_path


def unproject_depth_map_to_point_map(depth_map, extrinsic, intrinsic):
    depth = depth_map[..., 0] if depth_map.ndim == 4 else depth_map
    S, H, W = depth.shape
    y, x = np.meshgrid(np.arange(H), np.arange(W), indexing="ij")
    x = np.broadcast_to(x[None], (S, H, W)); y = np.broadcast_to(y[None], (S, H, W))
    fx = intrinsic[:, 0, 0][:, None, None]; fy = intrinsic[:, 1, 1][:, None, None]
    cx = intrinsic[:, 0, 2][:, None, None]; cy = intrinsic[:, 1, 2][:, None, None]
    cam = np.stack([(x - cx) / fx * depth, (y - cy) / fy * depth, depth], axis=-1)
    R = extrinsic[:, :3, :3]; t = extrinsic[:, :3, 3]
    return np.einsum("sij,shwj->shwi", np.transpose(R, (0, 2, 1)), cam - t[:, None, None, :])


def compute_camera_faces(cone_shape):
    faces_list = []
    num_vertices_cone = len(cone_shape.vertices)
    for face in cone_shape.faces:
        if 0 in face:
            continue
        v1, v2, v3 = face
        v1_offset, v2_offset, v3_offset = face + num_vertices_cone
        v1_offset_2, v2_offset_2, v3_offset_2 = face + 2 * num_vertices_cone
        faces_list.extend([
            (v1, v2, v2_offset), (v1, v1_offset, v3), (v3_offset, v2, v3),
            (v1, v2, v2_offset_2), (v1, v1_offset_2, v3), (v3_offset_2, v2, v3),
        ])
    faces_list += [(v3, v2, v1) for v1, v2, v3 in faces_list]
    return np.array(faces_list)
"""

SCENE_PATH = os.path.join(WORKERS_DIR, "scene_helper.py")
with open(SCENE_PATH, "w") as f:
    f.write(SCENE_SRC)
print(f"Wrote scene helper -> {SCENE_PATH}")

### What the accuracy numbers mean

- **Camera rotation error (degrees)** — how much the estimated camera
  orientation differs between the two precisions.
- **Camera translation error** — how far apart the two estimated camera
  positions are.
- **Depth error** — how much the predicted depth (distance from the camera) at
  each pixel differs, both in absolute terms and as a percentage.
- **Point-cloud drift** — how far the reconstructed 3D points move, reported
  both in real-world units and as a percentage of the scene's overall size (the
  percentage is what makes this comparable between a small scene and a large one).

All of these compare bf16 against that **same model's own mixed-precision
result**, on the same input — they measure how much bf16 changes the answer,
not whether either answer is "correct" against ground truth.

In [ ]:
# ── Write the per-model accuracy+glTF worker ────────────────────────────────
# One process per model (see the module-name-collision note above). Runs BOTH
# precisions (mixed then bf16) inside that one process — that's safe, unlike
# mixing two DIFFERENT models in one process, because there's no naming
# collision between two precision variants of the SAME model.
ACCURACY_WORKER_SRC = r"""
import os, sys, json, argparse, gc
import numpy as np

sys.path.insert(0, os.environ["NB_WORKERS_DIR"])
import scene_helper as SC

ap = argparse.ArgumentParser()
ap.add_argument("--model", choices=["vggt", "omega"], required=True)
ap.add_argument("--frames-dir", required=True)
ap.add_argument("--num-frames", type=int, required=True)
ap.add_argument("--gltf-dir", required=True)
ap.add_argument("--conf-thres-pct", type=float, default=50.0)
ap.add_argument("--max-points", type=int, default=1_500_000)
ap.add_argument("--seed", type=int, default=0)
args = ap.parse_args()

import torch

REPO_VGGT = os.environ["NB_REPO_VGGT"]
REPO_OMEGA = os.environ["NB_REPO_OMEGA"]
OMEGA_CKPT = os.environ["NB_OMEGA_CKPT"]
VGGT_HF = os.environ["NB_VGGT_HF"]

frame_paths = sorted(
    os.path.join(args.frames_dir, f) for f in os.listdir(args.frames_dir) if f.endswith(".png")
)[: args.num_frames]

dev = "cuda"
dtype = torch.bfloat16


def load_vggt():
    sys.path.insert(0, REPO_VGGT)
    from vggt.models.vggt import VGGT
    from vggt.utils.load_fn import load_and_preprocess_images
    from vggt.utils.pose_enc import pose_encoding_to_extri_intri
    # Public, ungated weights -- auto-downloads from https://huggingface.co/facebook/VGGT-1B
    model = VGGT.from_pretrained(VGGT_HF).eval().to(dev)
    images = load_and_preprocess_images(frame_paths).to(dev)
    def decode_camera(pose_enc, hw):
        extr, intr = pose_encoding_to_extri_intri(pose_enc, hw)
        return extr[0].float().cpu().numpy(), intr[0].float().cpu().numpy()
    def fwd(model_, images_b, mode):
        with torch.no_grad():
            if mode == "mixed":
                with torch.autocast(dev, dtype=dtype):
                    tok, ps = model_.aggregator(images_b)
                with torch.autocast(dev, enabled=False):
                    pose = model_.camera_head(tok)[-1]
                    depth, dconf = model_.depth_head(tok, images=images_b, patch_start_idx=ps)
            else:
                with torch.autocast(dev, dtype=dtype):
                    tok, ps = model_.aggregator(images_b)
                    pose = model_.camera_head(tok)[-1]
                    depth, dconf = model_.depth_head(tok, images=images_b, patch_start_idx=ps)
        return dict(pose_enc=pose, depth=depth, depth_conf=dconf)
    return model, images, decode_camera, fwd


def load_omega():
    sys.path.insert(0, REPO_OMEGA)
    from vggt_omega.models import VGGTOmega
    from vggt_omega.utils.load_fn import load_and_preprocess_images
    from vggt_omega.utils.pose_enc import encoding_to_camera
    model = VGGTOmega().to(dev).eval()
    # GATED checkpoint -- request access at https://huggingface.co/facebook/VGGT-Omega,
    # then download vggt_omega_1b_512.pt there once approved.
    model.load_state_dict(torch.load(OMEGA_CKPT, map_location="cpu", weights_only=False), strict=False)
    images = load_and_preprocess_images(frame_paths, image_resolution=512).to(dev)
    def decode_camera(pose_enc, hw):
        extr, intr = encoding_to_camera(pose_enc, hw)
        return extr[0].float().cpu().numpy(), intr[0].float().cpu().numpy()
    def fwd(model_, images_b, mode):
        with torch.no_grad():
            if mode == "mixed":
                with torch.autocast(dev, dtype=dtype):
                    tok, ps = model_.aggregator(images_b)
                with torch.autocast(dev, enabled=False):
                    pose = model_.camera_head(tok, patch_token_start=ps)
                    depth, dconf = model_.dense_head(tok, images=images_b, patch_token_start=ps)
            else:
                with torch.autocast(dev, dtype=dtype):
                    tok, ps = model_.aggregator(images_b)
                    pose = model_.camera_head(tok, patch_token_start=ps)
                    depth, dconf = model_.dense_head(tok, images=images_b, patch_token_start=ps)
        return dict(pose_enc=pose, depth=depth, depth_conf=dconf)
    return model, images, decode_camera, fwd


loader = load_vggt if args.model == "vggt" else load_omega
model, images, decode_camera, fwd = loader()
images_b = images.unsqueeze(0)
H, W = images.shape[-2:]
nparam = sum(p.numel() for p in model.parameters())

def postprocess(preds):
    extr, intr = decode_camera(preds["pose_enc"].float(), (H, W))
    depth = preds["depth"][0].float().cpu().numpy()
    if depth.ndim == 4:
        depth = depth[..., 0]
    dconf = preds["depth_conf"][0].float().cpu().numpy()
    world = SC.unproject_depth_map_to_point_map(depth, extr, intr)
    return dict(extr=extr, intr=intr, depth=depth, depth_conf=dconf, world=world)


def rot_deg(Ra, Rb):
    R = Ra @ np.transpose(Rb, (0, 2, 1))
    tr = np.clip((np.trace(R, axis1=1, axis2=2) - 1) / 2, -1, 1)
    return np.degrees(np.arccos(tr))


def accuracy(ref, new):
    a = new["depth"].reshape(-1).astype(np.float64)
    b = ref["depth"].reshape(-1).astype(np.float64)
    mk = np.isfinite(a) & np.isfinite(b)
    a, b = a[mk], b[mk]
    ae = np.abs(a - b)
    rel = ae / np.maximum(np.abs(b), 1e-6)
    d = np.linalg.norm(new["world"] - ref["world"], axis=-1).reshape(-1)
    d = d[np.isfinite(d)]
    wr = ref["world"].reshape(-1, 3)
    ext = float(np.linalg.norm(np.nanpercentile(wr, 95, axis=0) - np.nanpercentile(wr, 5, axis=0)))
    return dict(
        camera_rot_deg_mean=float(rot_deg(ref["extr"][:, :, :3], new["extr"][:, :, :3]).mean()),
        camera_rot_deg_max=float(rot_deg(ref["extr"][:, :, :3], new["extr"][:, :, :3]).max()),
        camera_trans_l2_mean=float(np.linalg.norm(ref["extr"][:, :, 3] - new["extr"][:, :, 3], axis=1).mean()),
        depth_mae=float(ae.mean()), depth_rel_mean_pct=float(rel.mean() * 100),
        points_l2_mean=float(d.mean()), points_l2_p95=float(np.percentile(d, 95)),
        scene_extent=ext, points_l2_mean_pct_extent=float(100 * d.mean() / max(ext, 1e-9)),
    )


result = dict(model=args.model, params_billion=nparam / 1e9, input_hw=[int(H), int(W)],
              n_frames=len(frame_paths), variants={})
post = {}
mem_by_mode = {}
for mode, images_b_mode in (("mixed", images_b), ("bf16", images_b.to(dtype))):
    if mode == "bf16":
        model = model.to(dtype)
    import torch as _torch
    # Warmup pass (discarded): whichever mode runs first pays a one-time cost
    # for CUDA kernel selection/compilation. Without this, that cost would show
    # up as inflated peak memory for "mixed" (which runs first) but not "bf16"
    # (which benefits from the aggregator already being warmed up) -- exactly
    # the asymmetric mixed-vs-bf16 gap this fix removes.
    _ = fwd(model, images_b_mode, mode)
    _torch.cuda.synchronize()
    _torch.cuda.reset_peak_memory_stats()
    mem_before = _torch.cuda.memory_allocated() / 1e9
    preds = fwd(model, images_b_mode, mode)
    _torch.cuda.synchronize()
    mem_by_mode[mode] = dict(
        model_resident_gb=mem_before,
        peak_allocated_gb=_torch.cuda.max_memory_allocated() / 1e9,
        peak_reserved_gb=_torch.cuda.max_memory_reserved() / 1e9,
    )
    post[mode] = postprocess(preds)
    gc.collect(); _torch.cuda.empty_cache()

result["variants"]["mixed"] = mem_by_mode["mixed"]
result["variants"]["bf16"] = mem_by_mode["bf16"]
result["accuracy_bf16_vs_mixed"] = accuracy(post["mixed"], post["bf16"])

# glTF export for both variants — identical filtering, per scene_helper.build_scene
images_np = images.float().cpu().numpy()
for mode in ("mixed", "bf16"):
    p = post[mode]
    sc, n_kept, n_conf = SC.build_scene(
        p["world"], p["depth_conf"], images_np, p["extr"],
        args.conf_thres_pct, args.max_points, seed=args.seed)
    gltf_path = os.path.join(args.gltf_dir, f"{args.model}_{mode}.gltf")
    g, b = SC.export_gltf_file(sc, gltf_path)
    result["variants"][mode]["gltf"] = g
    result["variants"][mode]["glb"] = b
    result["variants"][mode]["points_in_file"] = n_kept

print("RESULT_JSON:" + json.dumps(result))
"""

ACC_WORKER_PATH = os.path.join(WORKERS_DIR, "worker_accuracy.py")
with open(ACC_WORKER_PATH, "w") as f:
    f.write(ACCURACY_WORKER_SRC)
print(f"Wrote accuracy worker -> {ACC_WORKER_PATH}")

In [ ]:
# ── Run Part 2 for both models ──────────────────────────────────────────
GLTF_DIR = os.path.join(OUT_DIR, "gltf")
os.makedirs(GLTF_DIR, exist_ok=True)
ACCURACY_NUM_FRAMES = 12 if QUICK_TEST else 35   # SHARED.num_frames in the original pipeline

accuracy_results = {}
for model in ("vggt", "omega"):
    env = os.environ.copy()
    env.update(NB_REPO_VGGT=REPO_VGGT, NB_REPO_OMEGA=REPO_OMEGA,
               NB_OMEGA_CKPT=OMEGA_CHECKPOINT, NB_VGGT_HF=VGGT_HF_REPO,
               NB_WORKERS_DIR=WORKERS_DIR)
    cmd = [_sys.executable, ACC_WORKER_PATH, "--model", model,
           "--frames-dir", FRAMES_DIR, "--num-frames", str(ACCURACY_NUM_FRAMES),
           "--gltf-dir", GLTF_DIR]
    log(f"=== Accuracy/glTF: {model} ({ACCURACY_NUM_FRAMES} frames) ===")
    proc = subprocess.run(cmd, capture_output=True, text=True, timeout=900, env=env)
    line = next((l for l in proc.stdout.splitlines() if l.startswith("RESULT_JSON:")), None)
    if line is None:
        print(proc.stderr[-3000:])
        raise RuntimeError(f"{model} accuracy worker failed — see stderr above")
    accuracy_results[model] = json.loads(line[len("RESULT_JSON:"):])
    log(f"  {model}: done")

print("Part 2 complete.")

In [ ]:
# ── Display accuracy + resource + glTF results ──────────────────────────────
acc_rows = []
for model in ("vggt", "omega"):
    r = accuracy_results[model]
    a = r["accuracy_bf16_vs_mixed"]
    acc_rows.append(dict(
        model=labels[model],
        camera_rot_deg_mean=round(a["camera_rot_deg_mean"], 4),
        camera_trans_l2_mean=round(a["camera_trans_l2_mean"], 5),
        depth_rel_mean_pct=round(a["depth_rel_mean_pct"], 3),
        point_drift_pct_extent=round(a["points_l2_mean_pct_extent"], 3),
    ))
pd.DataFrame(acc_rows)

In [ ]:
resource_rows = []
for model in ("vggt", "omega"):
    r = accuracy_results[model]
    for mode in ("mixed", "bf16"):
        v = r["variants"][mode]
        resource_rows.append(dict(
            model=labels[model], precision=mode,
            model_resident_gb=round(v["model_resident_gb"], 3),
            peak_allocated_gb=round(v["peak_allocated_gb"], 3),
            peak_reserved_gb=round(v["peak_reserved_gb"], 3),
            points_in_gltf=v["points_in_file"],
            gltf_path=v["gltf"],
        ))
pd.DataFrame(resource_rows)

### A couple of notes on Part 2

- **Both models use the same number of frames**, so the comparison is fair —
  changing `ACCURACY_NUM_FRAMES` changes what's being measured (a bigger scene),
  not just how precisely it's measured.
- **The glTF/`.glb` files land under `OUT_DIR/gltf/`** — open them in any glTF
  viewer (for example [gltf-viewer.donmccurdy.com](https://gltf-viewer.donmccurdy.com/),
  or Blender) to visually compare `{model}_mixed.gltf` against `{model}_bf16.gltf`
  side by side.
- **The accuracy numbers are relative, not absolute** — a "0.05° rotation
  error" means the bf16 and mixed-precision results agree with each other to
  within 0.05°, not that either one is the objectively correct answer.

---
## Where everything lands

```
OUT_DIR/
├── frames_pool/              shared input frames (both parts use this)
├── workers/                  generated worker scripts (this notebook writes these itself)
├── results/
│   ├── notebook_results.json   every individual measurement from Part 1
│   └── notebook_run.log        a human-readable log of the Part 1 run
└── gltf/
    ├── vggt_mixed.gltf(+.glb)    open these in a glTF viewer to compare
    ├── vggt_bf16.gltf(+.glb)
    ├── omega_mixed.gltf(+.glb)
    └── omega_bf16.gltf(+.glb)
```

## Key takeaways

- **bf16 (true weight-cast) uses meaningfully less GPU memory** than the
  standard mixed-precision mode for both models, which directly translates into
  being able to process more frames before running out of memory.
- **The accuracy cost of bf16 is small** — camera pose, depth, and point-cloud
  differences between bf16 and mixed precision are consistently small across
  both models.
- **The memory benefit of bf16 shrinks as you process more frames at once** —
  most of the savings come from the model's weights taking up less space, and
  that's a fixed amount regardless of frame count, while the memory used by the
  frames themselves keeps growing. So bf16 helps most when memory is tight to
  begin with.
- **Speed differences between the two precisions are minor** — most of the time
  is spent in the backbone (aggregator), which is the same computational cost
  either way; the meaningful bf16 speedup shows up in the smaller output heads.